In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import random
from sklearn.metrics import confusion_matrix

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Transformacje danych

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),(0.5, 0.5, 0.5))])

## CIFAR10 dataset

In [4]:
train_dataset = torchvision.datasets.CIFAR10(root='./data',train=True,download=True,transform=transform)
train_loader = DataLoader(train_dataset,batch_size=64,shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data',train=False,download=True,transform=transform)
test_loader = DataLoader(test_dataset,batch_size=64,shuffle=False)

100%|██████████| 170M/170M [00:10<00:00, 15.5MB/s] 


## Definicja CNN

In [5]:
class CIFAR10CNN(nn.Module):
    def __init__(self):
        super(CIFAR10CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2,stride=2)
        
        self.conv2 = nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2,stride=2)
        # self.dropout = nn.Dropout(0.2) # usuwanie losowo danych 
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)
    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        # x = self.dropout(x)
        x = self.fc2(x)
        return x

## Tworzenie modelu

In [6]:
model = CIFAR10CNN().to(device)

## Loss function i optimizer

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)

## Training loop

In [8]:
epochs = 30

for epoch in tqdm(range(epochs)):
    model.train()
    running_loss = 0.0

    for i, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        optimizer.zero_grad()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

        if i % 100 == 99:
            print(
                f'Epoch {epoch+1}, '
                f'Batch {i+1}, '
                f'Loss: {running_loss / 100:.4f}')
            running_loss = 0.0

  0%|          | 0/30 [00:00<?, ?it/s]

Epoch 1, Batch 100, Loss: 1.8030
Epoch 1, Batch 200, Loss: 1.4681
Epoch 1, Batch 300, Loss: 1.3631
Epoch 1, Batch 400, Loss: 1.2780
Epoch 1, Batch 500, Loss: 1.2226
Epoch 1, Batch 600, Loss: 1.1508
Epoch 1, Batch 700, Loss: 1.1279


  3%|▎         | 1/30 [00:43<20:58, 43.38s/it]

Epoch 2, Batch 100, Loss: 0.9711
Epoch 2, Batch 200, Loss: 0.9758
Epoch 2, Batch 300, Loss: 0.9903
Epoch 2, Batch 400, Loss: 0.9427
Epoch 2, Batch 500, Loss: 0.9347
Epoch 2, Batch 600, Loss: 0.9026
Epoch 2, Batch 700, Loss: 0.8792


  7%|▋         | 2/30 [01:31<21:30, 46.08s/it]

...

 93%|█████████▎| 28/30 [19:03<01:22, 41.34s/it]

Epoch 29, Batch 100, Loss: 0.0294
Epoch 29, Batch 200, Loss: 0.0310
Epoch 29, Batch 300, Loss: 0.0245
Epoch 29, Batch 400, Loss: 0.0301
Epoch 29, Batch 500, Loss: 0.0500
Epoch 29, Batch 600, Loss: 0.0540
Epoch 29, Batch 700, Loss: 0.0432


 97%|█████████▋| 29/30 [19:49<00:42, 42.82s/it]

Epoch 30, Batch 100, Loss: 0.0402
Epoch 30, Batch 200, Loss: 0.0301
Epoch 30, Batch 300, Loss: 0.0317
Epoch 30, Batch 400, Loss: 0.0337
Epoch 30, Batch 500, Loss: 0.0403
Epoch 30, Batch 600, Loss: 0.0347
Epoch 30, Batch 700, Loss: 0.0399


100%|██████████| 30/30 [20:28<00:00, 40.95s/it]


In [9]:
torch.save(model.state_dict(),'model_CNN_30.pth')